# Directional Asymmetry Investigation: Melbourne→Hobart vs Hobart→Melbourne

This notebook investigates why Hobart→Melbourne (R²=0.55) performs well while Melbourne→Hobart (R²=0.08) performs poorly. Low flight volume was identified in notebook 6b. However, there may be another underlying problem present; because Melbourne <-> Hobart route pair has more than 50 monthly flights, yet Melbourne -> Hobart prediction performs poorly. 

**Answer:** This is largely due to **year-specific randomness** in the test set, not a fundamental difference between the routes.

**Key Finding:**
- 2019 was an anomalous year for Melbourne→Hobart (lag1 correlation collapsed to 0.04)
- Since 2019 is in the test set, this single year significantly lowers Melbourne->Hobart's reported test R²
- Cross-validation shows both routes have similar average performance with high variance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Plotting style
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

%matplotlib inline

In [ ]:
# Load data
df = pd.read_csv('../data/processed/ml_training_data_multiroute.csv')
df['year'] = df['year'].astype(int)
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['airline_route'] = df['airline'] + '_' + df['route']

# Sort for lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

# Create lag features
df['delay_rate_lag1'] = df.groupby('airline_route')['delay_rate'].shift(1)

print(f"Dataset: {len(df)} records")
print(f"Routes: {df['route'].nunique()}")

## 1. The Puzzle: Same Volume, Different R²

From notebook 6, the multi-route model shows a puzzling asymmetry:

In [ ]:
print("="*80)
print("THE PUZZLE: DIRECTIONAL ASYMMETRY")
print("="*80)

print("\n1. BASIC COMPARISON (Same Volume, Different R²):")
print("-"*60)
print(f"{'Metric':<30} {'Hobart→Mel':>15} {'Mel→Hobart':>15}")
print("-"*60)

metrics = {
    'Avg Volume': [103.0, 103.0],
    'Lag1 Corr (overall)': [0.70, 0.61],
    'Model R² (from notebook 6)': [0.55, 0.08],
}
for metric, vals in metrics.items():
    print(f"{metric:<30} {vals[0]:>15.2f} {vals[1]:>15.2f}")

print("\n→ Same volume, same airlines, same weather... why the huge R² gap?")

## 2. Year-by-Year Lag1 Correlation Analysis

Let's look at lag1 correlation by year to see if there are temporal patterns:

In [ ]:
print("\n2. YEAR-BY-YEAR LAG1 CORRELATION (The Key Insight):")
print("-"*60)
print(f"{'Year':<10} {'Hobart→Mel':>15} {'Mel→Hobart':>15} {'Δ':>10}")
print("-"*60)

years_to_check = [2018, 2019, 2023, 2024, 2025]
for year in years_to_check:
    vals = []
    for route in ['Hobart_Melbourne', 'Melbourne_Hobart']:
        route_data = df[(df['route'] == route) & (df['year'] == year)].dropna(subset=['delay_rate_lag1'])
        if len(route_data) > 10:
            corr = route_data['delay_rate'].corr(route_data['delay_rate_lag1'])
            vals.append(corr)
        else:
            vals.append(float('nan'))
    
    test_marker = " ← TEST" if year in [2019, 2025] else ""
    delta = vals[0] - vals[1] if not (np.isnan(vals[0]) or np.isnan(vals[1])) else float('nan')
    print(f"{year:<10} {vals[0]:>15.3f} {vals[1]:>15.3f} {delta:>10.3f}{test_marker}")

print("\n⚠️  2019: Melbourne→Hobart lag1 correlation collapsed to 0.04!")
print("   This single year dragged down the test set performance.")

## 3. Train vs Test Set Lag1 Correlation

The test set includes 2019 and 2025. Let's compare lag1 correlation in train vs test:

In [ ]:
print("\n3. TEST SET PERFORMANCE (2019 + 2025):")
print("-"*60)

test_years = [2019, 2025]

for route in ['Hobart_Melbourne', 'Melbourne_Hobart']:
    route_data = df[df['route'] == route]
    train = route_data[~route_data['year'].isin([2019, 2020, 2021, 2022, 2025])]
    test = route_data[route_data['year'].isin(test_years)]
    
    train_clean = train.dropna(subset=['delay_rate_lag1'])
    test_clean = test.dropna(subset=['delay_rate_lag1'])
    
    train_corr = train_clean['delay_rate'].corr(train_clean['delay_rate_lag1'])
    test_corr = test_clean['delay_rate'].corr(test_clean['delay_rate_lag1'])
    
    print(f"\n{route}:")
    print(f"  Train lag1 corr: {train_corr:.4f}")
    print(f"  Test lag1 corr:  {test_corr:.4f}  (Δ = {test_corr - train_corr:+.4f})")

print("\n→ Melbourne→Hobart lag1 dropped from 0.61 to 0.33 in test set!")
print("   This explains the poor R² - the model learned lag1 is predictive,")
print("   but in the test set (especially 2019), lag1 stopped working.")

## 4. Cross-Validation: Both Routes Have High Variance

Using TimeSeriesSplit CV to get a more robust performance estimate:

In [ ]:
print("\n4. CROSS-VALIDATION (High Variance for Both Routes):")
print("-"*60)

cv_results = {}

for route in ['Hobart_Melbourne', 'Melbourne_Hobart']:
    route_data = df[df['route'] == route].dropna(subset=['delay_rate_lag1'])
    route_data = route_data[~route_data['year'].isin([2020, 2021, 2022])].sort_values('year_month_dt')
    
    X = route_data[['delay_rate_lag1']].values
    y = route_data['delay_rate'].values
    
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        
        model = Ridge(alpha=1.0)
        model.fit(X_train_s, y_train)
        scores.append(r2_score(y_test, model.predict(X_test_s)))
    
    cv_results[route] = scores
    
    print(f"\n{route}:")
    print(f"  Fold R² scores: {[f'{s:.3f}' for s in scores]}")
    print(f"  Mean ± Std: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

print("\n→ Both routes show HIGH VARIANCE across folds.")
print("   The 0.55 vs 0.08 difference from notebook 6 is largely")
print("   due to which years ended up in the test set.")

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Year-by-year lag1 correlation comparison
ax = axes[0]
years = [2018, 2019, 2023, 2024, 2025]
hba_mel = []
mel_hba = []

for year in years:
    for route, lst in [('Hobart_Melbourne', hba_mel), ('Melbourne_Hobart', mel_hba)]:
        route_data = df[(df['route'] == route) & (df['year'] == year)].dropna(subset=['delay_rate_lag1'])
        if len(route_data) > 10:
            lst.append(route_data['delay_rate'].corr(route_data['delay_rate_lag1']))
        else:
            lst.append(np.nan)

x = np.arange(len(years))
width = 0.35
ax.bar(x - width/2, hba_mel, width, label='Hobart→Melbourne', color='#3498db')
ax.bar(x + width/2, mel_hba, width, label='Melbourne→Hobart', color='#e74c3c')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Lag1 Correlation')
ax.set_title('Lag1 Correlation by Year and Direction')
ax.set_xticks(x)
ax.set_xticklabels([f'{y}\n(TEST)' if y in [2019, 2025] else str(y) for y in years])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Highlight 2019
ax.annotate('← 2019 anomaly', xy=(1 + width/2, 0.04), xytext=(1.5, 0.2),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')

# Plot 2: CV fold distribution
ax = axes[1]
cv_hba_mel = cv_results['Hobart_Melbourne']
cv_mel_hba = cv_results['Melbourne_Hobart']

ax.boxplot([cv_hba_mel, cv_mel_hba], labels=['Hobart→Melbourne', 'Melbourne→Hobart'])
ax.scatter([1]*5, cv_hba_mel, alpha=0.6, s=50, color='#3498db')
ax.scatter([2]*5, cv_mel_hba, alpha=0.6, s=50, color='#e74c3c')
ax.set_ylabel('R² Score')
ax.set_title('Cross-Validation R² Distribution\n(5 folds each)')
ax.grid(True, alpha=0.3, axis='y')

# Annotate the problematic fold
min_score = min(cv_mel_hba)
ax.annotate(f'Fold with R²={min_score:.3f}', xy=(2, min_score), xytext=(2.2, 0.15),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

plt.tight_layout()
plt.show()

## 6. Conclusion

The Hobart→Melbourne vs Melbourne→Hobart performance gap (0.55 vs 0.08) is not
due to a fundamental difference between the routes. They have the same variables: aircraft, weather (just swapped departure/arrival), airlines and volume above threshold (~103 flights/month).

The difference is due to an anomaly in one specific year:
1. 2019 was an anomalous year for Melbourne -> Hobart
   - Lag1 correlation collapsed to 0.04 (essentially random)
   - The opposite route, Hobart→Melbourne, maintained a lag1 correlation of 0.44.
2. The test set includes 2019, which disproportionately hurt Melbourne -> Hobart
3. Cross-validation shows both routes have similar average performance:
   - Hobart→Melbourne: 0.34 ± 0.15
   - Melbourne→Hobart: 0.28 ± 0.16
4. Both routes show HIGH VARIANCE across folds, meaning performance is sensitive to which time periods are used for testing

## 7. Next step

At the current stage of development, the sudden drop in correlation in 2019 is treated as an anomaly and will be excluded from the routes covered by the model. Future work may include random train-test splits or exclude anomalous years from the training process to solve this problem.

For now, the project moves on to implement and assess the model performance with low-volume airline-routes removed from the training dataset.